In [0]:
from pyspark.sql.functions import current_timestamp, from_unixtime, col
from pyspark.sql.functions import lit, from_json, schema_of_json
from pyspark.sql.functions import explode_outer

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType

schema = StructType([
    StructField("name", StringType()),
    StructField("timezone", IntegerType()),
    StructField("dt", IntegerType()),
    StructField("main", StructType([
        StructField("temp", DoubleType()),
        StructField("feels_like", DoubleType()),
        StructField("sea_level", IntegerType()),
        StructField("grnd_level", IntegerType()),
        StructField("temp_min", DoubleType()),
        StructField("temp_max", DoubleType()),
        StructField("humidity", IntegerType()),
        StructField("pressure", IntegerType()),
    ])),
    StructField("weather", ArrayType(StructType([
        StructField("id", IntegerType()),
        StructField("main", StringType()),
        StructField("description", StringType()),
        StructField("icon", StringType()),
    ]))),
    StructField("wind", StructType([
        StructField("speed", DoubleType()),
        StructField("deg", IntegerType()),
        StructField("gust", DoubleType()),
    ])),
    StructField("coord", StructType([
        StructField("lat", DoubleType()),
        StructField("lon", DoubleType()),
    ])),
    StructField("sys", StructType([
        StructField("country", StringType()),
        StructField("sunrise", IntegerType()),
        StructField("sunset", IntegerType()),
    ])),
    StructField("clouds", StructType([
        StructField("all", IntegerType()),
    ])),
])

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import from_json, from_unixtime, to_json
from datetime import datetime

try:
    last_ingestion = spark.table("dev.weather.silver_weather").agg(F.max("ingestion_time")).collect()[0][0]
except Exception as error:
    last_ingestion = datetime(1970, 1, 1)

df_bronze = spark.read.table("dev.weather.bronze_weather") \
    .filter(col("ingestion_time") > last_ingestion) \
    .dropDuplicates(["raw_json"])
print(f"Anzahl Einträge: {df_bronze.count()}")

df_silver = df_bronze \
    .withColumn("data", from_json("raw_json", schema)) \
    .withColumn("weather_exploded", explode_outer("data.weather")) \
    .withColumn("_rescued_data", to_json("data")) \
    .select(
        col("ingestion_time"),
        col("data.name"),
        col("data.coord.lat"),
        col("data.coord.lon"),
        col("data.sys.country"),
        col("data.timezone"),
        from_unixtime("data.dt").cast("timestamp").alias("datetime"),
        from_unixtime("data.sys.sunrise").cast("timestamp").alias("sunrise_at"),
        from_unixtime("data.sys.sunset").cast("timestamp").alias("sunset_at"),
        col("data.main.temp").alias("temp_c"),
        col("data.main.feels_like").alias("feels_like_temp_c"),
        col("data.main.temp_min").alias("temp_min_c"),
        col("data.main.temp_max").alias("temp_max_c"),
        col("data.main.humidity").alias("humidity_pct"),
        col("data.main.pressure").alias("pressure_hpa"),
        col("data.main.sea_level").alias("sea_level_hpa"),
        col("data.main.grnd_level").alias("grnd_level_hpa"),
        col("data.wind.speed").alias("wind_speed_ms"),
        col("data.wind.deg").alias("wind_deg"),
        col("data.wind.gust").alias("wind_gust_ms"),
        col("data.clouds.all").alias("cloud_pct"),
        col("weather_exploded.id").alias("weather_id"),
        col("weather_exploded.main").alias("weather_main"),
        col("weather_exploded.description").alias("weather_desc"),
        col("weather_exploded.icon").alias("weather_icon"),
        col("_rescued_data")
    )

display(df_silver)
df_silver.printSchema()

df_silver.write.format("delta")\
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("dev.weather.silver_weather")